# 20 - Grasping And Manipulation

## What / Why / How

**What we are trying to do:** Build first principles for grasping: friction cones, antipodal contacts, and pick-place state machines.

**Why this matters:** Manipulation is not only arm motion. It also needs contact reasoning, perception, sequencing, and recovery behavior.

**How we will do it:** Check whether forces fit a friction cone, score simple contact pairs, and step through a pick-place finite-state machine.

## Goal

Grasping is where geometry, contact, planning, and perception meet.

You will model:

- A friction cone
- Antipodal grasp intuition
- A pick-place state machine

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
def within_friction_cone(contact_normal, force, mu):
    n = contact_normal / np.linalg.norm(contact_normal)
    f = force / np.linalg.norm(force)
    normal_component = max(0.0, f @ n)
    tangential = np.linalg.norm(f - normal_component * n)
    return tangential <= mu * normal_component

normal = np.array([1.0, 0.0])
test_forces = [np.array([1.0, 0.1]), np.array([1.0, 0.8]), np.array([-1.0, 0.0])]
for force in test_forces:
    print(force, "stable?", within_friction_cone(normal, force, mu=0.5))

In [ ]:
object_points = np.array([[np.cos(a), np.sin(a)] for a in np.linspace(0, 2*np.pi, 64, endpoint=False)])

def grasp_quality(i, j):
    p1, p2 = object_points[i], object_points[j]
    distance = np.linalg.norm(p1 - p2)
    center_alignment = abs((p1 + p2).sum())
    return distance - 0.2 * center_alignment

candidates = []
for i in range(len(object_points)):
    for j in range(i + 1, len(object_points)):
        candidates.append((grasp_quality(i, j), i, j))
best = max(candidates)
print("best quality, contacts:", best)
print("contact points:", object_points[best[1]], object_points[best[2]])

In [ ]:
states = ["approach", "pregrasp", "close_gripper", "lift", "move_to_place", "open_gripper", "retreat"]
state_index = 0
log = []
for tick in range(14):
    state = states[state_index]
    log.append((tick, state))
    if tick % 2 == 1 and state_index < len(states) - 1:
        state_index += 1
print(log)

## Exercises

1. Add a failure state for missed grasp.
2. Add a force threshold to decide when the gripper has closed.
3. Explain what perception must estimate before grasp planning can start.